# 06. Case study — Review an invoice before approval

Build a small assistant that can look up an invoice or compare it with the purchase order and delivery receipt. Run the cells in order. Use the [module setup](README.md#setup) and leave Ollama running with `qwen3.5:4b`. The fictional records are included below. This notebook calls Ollama directly, so the sales API can stay off.

## 1. Work out the policy

Maya works in accounts payable. She asks, “Check INV-1043 before I send it for approval.” An **invoice** says what the supplier billed. A **purchase order** says what the company agreed to buy. A **delivery receipt** records what arrived.

| Invoice | Billed | Ordered | Received |
|---|---|---|---|
| INV-1042 | 10 units, ₹4,500 | 10 units at ₹450 | 10 units |
| INV-1043 | 6 units, ₹7,800 | 6 units at ₹1,200 | 4 units |
| INV-1044 | 5 units, ₹1,250 | 5 units at ₹250 | Receipt missing |

Decide which invoice is ready and what Maya should investigate in the other two.

<details><summary>Check your answer</summary>

INV-1042 matches. INV-1043 bills ₹600 above the order total and bills six units when four arrived. INV-1044 needs the receipt.

</details>

Each example has one item line and one purchase order. Use whole rupees with no tax or shipping. Python checks the supplier, quantity, and amount against the order, then compares the billed and received quantities. The assistant reports whether a person can review the invoice for approval. The approval itself stays with Maya.

## 2. Implement the checks in Python

The invoice includes the purchase order ID. Use it to find the matching order and receipt. Run the cell to inspect the three records for INV-1043.

In [ ]:
invoices = {
    "INV-1042": {"po": "PO-71", "supplier": "Paper House", "units": 10, "amount": 4500},
    "INV-1043": {"po": "PO-72", "supplier": "Desk Supply", "units": 6, "amount": 7800},
    "INV-1044": {"po": "PO-73", "supplier": "Pack Works", "units": 5, "amount": 1250},
}

orders = {
    "PO-71": {"supplier": "Paper House", "units": 10, "unit_price": 450},
    "PO-72": {"supplier": "Desk Supply", "units": 6, "unit_price": 1200},
    "PO-73": {"supplier": "Pack Works", "units": 5, "unit_price": 250},
}

received_units = {"PO-71": 10, "PO-72": 4}

invoice = invoices["INV-1043"]
print("Invoice", invoice)
print("Purchase order", orders[invoice["po"]])
print("Units received", received_units.get(invoice["po"]))

Multiply the ordered quantity by the unit price, then compare the result with the bill. Collect each difference in a list so the review can report more than one issue at a time.

In [ ]:
def review_records(invoice_id):
    invoice = invoices.get(invoice_id)
    if invoice is None:
        return {"invoice_id": invoice_id, "status": "not_found"}

    po_id = invoice["po"]
    order = orders.get(po_id)
    received = received_units.get(po_id)
    expected_amount = None
    issues = []

    if order is None:
        issues.append(f"Purchase order {po_id} is missing.")
    else:
        expected_amount = order["units"] * order["unit_price"]
        if invoice["supplier"] != order["supplier"]:
            issues.append("The supplier differs from the purchase order.")
        if invoice["units"] != order["units"]:
            issues.append("The billed quantity differs from the purchase order.")
        if invoice["amount"] != expected_amount:
            issues.append(
                f"Invoice bills ₹{invoice['amount']:,}. "
                f"The purchase order totals ₹{expected_amount:,}."
            )

    # None means no receipt was found. A receipt with 0 units still exists.
    if received is None:
        issues.append(f"Delivery receipt for {po_id} is missing.")
    elif received != invoice["units"]:
        issues.append(f"Invoice bills {invoice['units']} units. {received} arrived.")

    return {
        "invoice_id": invoice_id,
        "status": "needs_review" if issues else "ready_for_approval",
        "evidence": {
            "purchase_order": po_id,
            "billed_amount": invoice["amount"],
            "ordered_amount": expected_amount,
            "billed_units": invoice["units"],
            "received_units": received,
        },
        "issues": issues,
    }


for invoice_id in ("INV-1042", "INV-1043", "INV-1044"):
    result = review_records(invoice_id)
    print(invoice_id, result["status"], result["issues"])

The matching invoice has an empty issue list. For INV-1043, the bill is $7,800 and the order totals $7,200. The receipt shows four of the six billed units. The review should report both issues. INV-1044 has no receipt, so the quantity is `None`. A recorded zero would mean a receipt exists and says zero units arrived.

## 3. Give the model two tools

The model can request `get_invoice` for the invoice's fields or `review_invoice` to check it against the other records. Both tools take an invoice ID. `bind_tools` sends the model their names, descriptions, and input schemas. [LangChain tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling)

In [ ]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama


@tool
def get_invoice(invoice_id: str) -> dict:
    """Look up an invoice's own fields, such as its supplier or billed amount."""
    invoice = invoices.get(invoice_id)
    if invoice is None:
        return {"invoice_id": invoice_id, "status": "not_found"}
    return {"invoice_id": invoice_id, "status": "details", **invoice}


@tool
def review_invoice(invoice_id: str) -> dict:
    """Review an invoice for approval by checking its order and delivery receipt."""
    return review_records(invoice_id)


tools = [get_invoice, review_invoice]
tools_by_name = {invoice_tool.name: invoice_tool for invoice_tool in tools}

model = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    reasoning=False,
    client_kwargs={"timeout": 45},
)
model_with_tools = model.bind_tools(tools)

instructions = """You help Maya with one invoice at a time.
For an invoice's fields or billed amount, choose get_invoice.
For checking an invoice before approval, choose review_invoice.
Call exactly one tool with the invoice ID supplied in the user message.
"""

Check that the question contains a single invoice ID. After the model chooses a tool, validate the tool name and invoice ID before running it. If the requested ID differs from the one Maya typed, stop and ask her to try again.

In [ ]:
import re


def format_answer(result):
    status = result["status"]
    invoice_id = result["invoice_id"]

    if status == "not_found":
        return f"Invoice {invoice_id} was not found."
    if status == "details":
        return (
            f"{invoice_id} from {result['supplier']} bills "
            f"{result['units']} units for ₹{result['amount']:,}."
        )
    if status == "needs_review":
        return f"{invoice_id} needs review. " + " ".join(result["issues"])
    return f"{invoice_id} matches the records and is ready for human approval."


def ask_invoice_assistant(question):
    # The sample invoice IDs use the pattern INV- followed by four digits.
    ids = set(re.findall(r"\bINV-[0-9]{4}\b", question.upper()))
    if len(ids) != 1:
        return {
            "status": "clarify",
            "tool": None,
            "answer": "Please give me one invoice ID, for example INV-1042.",
        }

    # There is exactly one ID in the set, so we can take it out.
    invoice_id = ids.pop()
    reply = model_with_tools.invoke([("system", instructions), ("human", question)])

    if len(reply.tool_calls) == 1:
        call = reply.tool_calls[0]
        chosen_tool = tools_by_name.get(call["name"])
        requested_id = str(call["args"].get("invoice_id", "")).strip().upper()
        if chosen_tool is not None and requested_id == invoice_id:
            result = chosen_tool.invoke({"invoice_id": invoice_id})
            return {**result, "tool": chosen_tool.name, "answer": format_answer(result)}

    return {
        "status": "tool_request_failed",
        "tool": None,
        "answer": "I couldn't verify that tool request. Please try again.",
    }

Run the lookup and review on the same invoice. Predict the tool name in each case. Notice that Python formats the reply directly from the tool result.

In [ ]:
for question in (
    "What amount is billed on INV-1043?",
    "Check INV-1043 before I send it for approval.",
):
    result = ask_invoice_assistant(question)
    print(question)
    print("Tool", result["tool"])
    print("Status", result["status"])
    print("Evidence", result.get("evidence", {}))
    print("Answer", result["answer"])
    print()

The lookup gives the billed amount. The review also shows the purchase order total and the delivered quantity. If Maya asks “Review it” in a new chat, the application needs to ask for an invoice ID.

## 4. Test the policy and the assistant

Call `review_records` directly to test the matching invoice, the mismatches, and the missing receipt. These checks use the sample data and can run without a model.

In [ ]:
assert review_records("INV-1042")["status"] == "ready_for_approval"

mismatch = review_records("INV-1043")
assert mismatch["status"] == "needs_review"
assert mismatch["evidence"]["ordered_amount"] == 7200
assert mismatch["evidence"]["received_units"] == 4
assert len(mismatch["issues"]) == 2

missing_receipt = review_records("INV-1044")
assert missing_receipt["status"] == "needs_review"
assert missing_receipt["evidence"]["received_units"] is None

print("PASS | Matching records, two mismatches, and a missing receipt")

Now call the full assistant. Check the tool choice, the structured status, and a relevant part of the reply. The Python checks tell you whether the review rules work. The full assistant checks also show whether a review request reaches the review tool.

In [ ]:
cases = [
    ("What amount is billed on INV-1043?", "get_invoice", "details", "₹7,800"),
    ("Check INV-1042 before approval.", "review_invoice", "ready_for_approval", "human approval"),
    ("Check INV-1043 before approval.", "review_invoice", "needs_review", "4 arrived"),
    ("Check INV-1044 before approval.", "review_invoice", "needs_review", "receipt"),
    ("Check INV-9999 before approval.", "review_invoice", "not_found", "was not found"),
    ("Can I send this invoice for approval?", None, "clarify", "one invoice ID"),
    ("Check INV-1042 and INV-1043.", None, "clarify", "one invoice ID"),
]

passed = 0
for question, expected_tool, expected_status, expected_text in cases:
    actual = ask_invoice_assistant(question)
    ok = (
        actual["tool"] == expected_tool
        and actual["status"] == expected_status
        and expected_text in actual["answer"]
    )
    passed += int(ok)
    print("PASS" if ok else "FAIL", "|", question)
    print(" ", actual["tool"], "|", actual["status"], "|", actual["answer"])

print(f"{passed}/{len(cases)} cases passed.")

The last two cases ask for clarification before calling a tool. If another case fails, inspect the selected tool and returned status first.

## 5. Update the receipt and rerun

The warehouse posts a receipt for five units on PO-73. Predict what changes for INV-1044. The next cell simulates the update and checks the result.

In [ ]:
# Simulate the warehouse posting the receipt.
received_units["PO-73"] = 5
try:
    updated = ask_invoice_assistant("Check INV-1044 before approval.")
    print(updated["answer"])
    assert updated["tool"] == "review_invoice"
    assert updated["status"] == "ready_for_approval"
    assert updated["evidence"]["received_units"] == 5
    print("PASS | The review uses the newly posted receipt.")
finally:
    # Restore the sample so earlier cells can be rerun.
    received_units.pop("PO-73", None)

**Extend the case.** If the business allows partial billing, write one case that should pass and one that should fail before editing the policy.

To expose the assistant to a website, wrap `ask_invoice_assistant` in a route using the pattern from [Lesson 04](04_sales_agent_api.md). Return the status, evidence, and answer so the client can display them separately. Finish with [Lesson 07](07_deploying_agentic_systems_on_aws.md) to see where the API and model could run.